[Course material - Sustain.Brussels - "Avdanced AI workflows with LLM" - 20/04/2026 - 22/04/2026](https://github.com/Yannael/gen-ai-sustain-brussels-2604).

In [ ]:
#!pip install openai

from openai import OpenAI
import json
import pandas as pd

client = OpenAI(api_key="...")


# Forcing GPT to return JSON Output

Sometimes, when you're using GPT in an application, it's helpful to get the response in a **structured format**, like **JSON**.  
This makes it easier for your program to read and use the result.

In this example, we tell GPT to return a JSON object by using the `response_format` parameter:


In [ ]:
response = client.chat.completions.create(
    model="gpt-5.4-nano",
    response_format={"type": "json_object"}, # Let us force the response to be JSON
    messages=[
        {
            "role": "system",
            "content": (
                "Analyze the customer's message and return a JSON object with two fields: 'tag' and 'sentiment'."
            )
        },
        {
            "role": "user",
            "content": "Why was my package not delivered yet?"
        }
    ]
)

print(response.choices[0].message.content)

{"tag":"order_delivery_status","sentiment":"negative"}


#### Tip: Be Clear About the Expected Format

When you use:

`response_format={"type": "json_object"}`

you are telling the OpenAI API that you want the response to be in JSON format.
However, the model also needs to be told in the prompt itself what kind of answer you expect.

✅ So it's important to include a clear instruction in the system message, like:
`"Analyze the customer's message and return a JSON object with two fields: 'tag' and 'sentiment'."
`

# Being Precise Helps Control the Output

To get reliable and structured answers from the LLM, it's important to **be as clear and specific as possible** in the system message.

In [ ]:
response = client.chat.completions.create(
        model="gpt-5.4-nano",
        response_format={"type": "json_object"},
        messages=[
            {
                "role": "system",
                "content": (
                    """
                    Analyze the customer's message and return a JSON object with 2 fields: 'tag' and 'sentiment'.

                    - 'tag' should describe the main topic of the message. It must be one of the following values: billing, technical_support, product_feedback, cancellation, other.
                    - 'sentiment' should describe the tone of the email and must be one of the following values: positive, neutral, negative.

                    Example output:
                    {
                        "tag": "<the tag of the message>",
                        "sentiment": "<the tone of the message>",
                    }
                    """
                )
            },
            {
                "role": "user",
                "content": "The billing was wrong, can you please correct it?"
            }
        ]
    )


print(response.choices[0].message.content)


{
  "tag": "billing",
  "sentiment": "negative"
}


We improve control over the response by:


* Defining exactly which fields we want in the JSON (tag, sentiment)
* Listing the allowed values for intent
* Giving a clear example of what the output should look like






# 📬 Classifying Many Emails Using a Function

Let us define a function called `def analyze_messages(input_message)` that takes one message and returns a **tag** and a **sentiment**. This makes it easy to use it for **lots of messages**.

In [ ]:
def analyze_message(input_message):
    response = client.chat.completions.create(
        model="gpt-5.4",
        response_format={"type": "json_object"},
        messages=[
            {
                "role": "system",
                "content": (
                    """
                    Analyze the customer's message and return a JSON object with 2 fields: 'tag' and 'sentiment'.

                    - 'tag' should describe the main topic of the message. It must be one of the following values: billing, technical_support, product_feedback, cancellation, other.
                    - 'sentiment' should describe the tone of the email and must be one of the following values: positive, neutral, negative.

                    Example output:
                    {
                        "tag": "<the tag of the message>",
                        "sentiment": "<the tone of the message>",
                    }
                    """
                )
            },
            {
                "role": "user",
                "content": input_message
            }
        ]
    )


    result = json.loads(response.choices[0].message.content)
    return result["tag"], result["sentiment"]


We just need to:
1. Put all our messages in a list.
2. Call the function on each message using a simple loop.
3. Store the results in a table (here, we use a `DataFrame` from the `pandas` library).

This illustrates how arrays of data can be processed and transformed using LLM calls.


In [ ]:
messages = [
    "I just wanted to say how much I love your new app design—great job!",
    "RThe biling was wrong",
    "My internet keeps disconnecting every few minutes, and it's really annoying.",
    "I’ve enjoyed the service, but I no longer need it. Please cancel my subscription."
]


res = pd.DataFrame(columns=["message","tag", "sentiment"])
for message in messages:
    tag, sentiment = analyze_message(message)
    new_row = pd.DataFrame({"message": [message],
                            "tag": [tag],
                            "sentiment": [sentiment],
                            })
    res = pd.concat([res, new_row], ignore_index=True)

res

,message,tag,sentiment
0,I just wanted to say how much I love your new ...,product_feedback,positive
1,RThe biling was wrong,billing,negative
2,My internet keeps disconnecting every few minu...,technical_support,negative
3,"I’ve enjoyed the service, but I no longer need...",cancellation,neutral


# ✏️ Practice Questions

1. **Update the list of tags**  
   Modify the system message so that the allowed tags are now:  
   `"billing"`, `"technical_support"`, `"product_feedback"`, `"shipping"`, `"cancellation"`, and `"other"`  
   ➕ This introduces a new category: `"shipping"`

2. **Add a third field: `urgency` (option)**  
   Extend the system message to request a new field in the JSON output: `"urgency"`  
   This field should describe how urgent the message is, with one of the following values:  
   `"high"`, `"medium"`, or `"low"`

   ✏️ For example, a message like  
   *"I’ve been overcharged and no one is replying to my emails!"*  
   would likely be:  
   `{"tag": "billing", "sentiment": "negative", "urgency": "high"}`